# Naive Bayes

We want to compute the posterior distribution $P(y|x)$, which is the probability of a class label $y$ given the input features $x$. Using Bayes' theorem, we can express this as:

$$P(y|x) = \frac{P(x|y) P(y)}{P(x)}$$

Where:
- $P(x|y)$ is the likelihood of the features given the class label.
- $P(y)$ is the prior probability of the class label.
- $P(x)$ is the evidence, which can be computed as $P(x) = \sum_{y} P(x|y) P(y)$. 

Since $P(x)$ is constant for all classes, we can ignore it when comparing the posterior probabilities for different classes. Therefore, we can simplify our decision rule to:

$$\hat{y} = \arg\max_y P(x|y) P(y)$$
This means that we will predict the class label that maximizes the product of the likelihood and the prior probability.



Assuming feature independence, we can further decompose the likelihood $P(x|y)$ into the product of the individual feature likelihoods:

$$P(x|y) = \prod_{i=1}^{n} P(x_i|y)$$

Where $x_i$ is the $i$-th feature. This assumption of feature independence is what gives the Naive Bayes classifier its name, as it is a "naive" assumption that may not hold in practice, but it often works well in many applications.

To avoid numerical underflow when multiplying many probabilities together, we can take the logarithm of the decision rule:

$$\hat{y} = \arg\max_y \log P(y) + \sum_{i=1}^{n} \log P(x_i|y)$$
This allows us to work with sums of logarithms instead of products of probabilities, which is more numerically stable.


## Training Process
To train a Naive Bayes classifier, we need to estimate the parameters of the model, which include the prior probabilities $P(y)$ and the likelihoods $P(x_i|y)$ for each feature and class label.

1. **Estimate Prior Probabilities**: We can estimate the prior probabilities $P(y)$ by counting the frequency of each class label in the training data and dividing by the total number of samples.
2. **Estimate Likelihoods**: We can estimate the likelihoods $P(x_i|y)$ by counting the frequency of each feature value for each class label in the training data and dividing by the total number of samples for that class label. Depending on the type of features (categorical or continuous), we may use different methods to estimate these probabilities, such as using a multinomial distribution for categorical features or a Gaussian distribution for continuous features.
3. **Smoothing**: To handle cases where a feature value does not occur in the training data for a particular class label (which would lead to zero probabilities), we can apply smoothing techniques such as Laplace smoothing, which adds a small constant to the counts to ensure that all probabilities are non-zero.



For a Gaussian Naive Bayes classifer, the likelihood

$$P(x_i|y) = \frac{1}{\sqrt{2\pi \sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$

Where $\mu_y$ and $\sigma_y^2$ are the mean and variance of the feature $x_i$ for class label $y$, which can be estimated from the training data.

## Numpy Implementation


In [ ]:
import numpy as np

In [ ]:
class GaussianNaiveBayes:
    def fit(self, X, y):
        """
        Args:
            X: numpy array of shape (N, D) - training data
            y: numpy array of shape (N,) - class labels 
        """
        N, D = X.shape
        self.classes = np.unique(y)
        C = len(self.classes)

        # initialize parameters
        self.priors = np.zeros(C) # P(y)
        self.means = np.zeros((C, D)) # mu_y
        self.variances = np.zeros((C, D)) # sigma_y^2
        
        for idx, c in enumerate(self.classes):
            X_c = X[y == c] # select samples of class c
            self.priors[idx] = X_c.shape[0] / N # P(y=c)
            self.means[idx, :] = np.mean(X_c, axis=0) # mu_y
            self.variances[idx, :] = np.var(X_c, axis=0) + 1e-9 # sigma_y^2
        
    
    def predict(self, X):
        """
        Args:
            X: numpy array of shape (M, D) - test data
        Returns:
            y_pred: numpy array of shape (M,) - predicted class labels
        """
        M, D = X.shape
        C = len(self.classes)
        log_probs = np.zeros((M, C)) # log P(y) + log P(x|y)

        for idx, c in enumerate(self.classes):
            # calculate log P(y=c)
            log_prior = np.log(self.priors[idx])
            # calculate log P(x|y=c) using Gaussian likelihood
            log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * self.variances[idx]))
            log_likelihood -= 0.5 * np.sum(((X - self.means[idx]) ** 2) / self.variances[idx], axis=1)
            log_probs[:, idx] = log_prior + log_likelihood

        y_pred = self.classes[np.argmax(log_probs, axis=1)]
        return y_pred